# Практична робота №4: пороги, вартість помилок і прийняття рішень

**Варіант:** 7 — Titanic, позитивний клас `AtRisk = 1` (пасажир не вижив), модель — логістична регресія з ЛР2.
Питання роботи: **яку дію виконати за прогнозом моделі, якщо помилки мають різні наслідки, а ресурси обмежені?**

In [1]:
from datetime import date
import json
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().parents[1]
WORK_DIR = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(WORK_DIR / "src"))

from src.titanic_workflow import RANDOM_STATE, load_titanic, make_cv, make_preprocessor, split_features_target, stratified_split
from decision_policy import DecisionPolicy

FIGURES = WORK_DIR / "figures"
RESULTS = WORK_DIR / "results"
CONFIGS = WORK_DIR / "configs"
for folder in (FIGURES, RESULTS, CONFIGS):
    folder.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)

## 1. Контекст рішення

**Навчальний сценарій.** Під час евакуації судна екіпаж має обмежену кількість людей для персонального супроводу пасажирів до шлюпок. Модель за даними реєстрації (клас, стать, вік, родина на борту, тариф, порт посадки — відомі з моменту посадки) оцінює ризик загибелі. Історичні дані Titanic використовуються як модель такої ситуації; числа нижче не є оцінкою реальної рятувальної системи.

| Питання | Відповідь |
|---|---|
| `y = 1` / `y = 0` | пасажир не вижив / вижив |
| Дія за `ŷ = 1` | пасажир включається до списку **пріоритетного супроводу** членом екіпажу |
| Дія за `ŷ = 0` | стандартна евакуація за загальною процедурою |
| Хто виконує дію | черговий офіцер формує список, члени екіпажу супроводжують |
| Коли доступні ознаки | з моменту посадки (реєстраційні дані) — до будь-якої події |
| Коли відома справжня мітка | лише після події; для моніторингу — за результатами навчальних тривог/інцидентів, із великою затримкою |

**Наслідки помилок:**

- **FP** — супровід отримав пасажир, який упорався б сам: витрачено ~15–20 хв роботи члена екіпажу, яких бракуватиме іншим. Вимірюється в «зайвих супроводах».
- **FN** — пасажир із реальним ризиком не потрапив у пріоритетний список і залишився без допомоги. Наслідок — потенційна загибель людини; грошима не вимірюється, тому в роботі використовується лише **відношення** `c_FN / c_FP` як сценарне припущення.

## 2. Фіксація моделі та протоколу

- Дані, ціль, split: як у ЛР2 (стратифіковано 80/20, `random_state=42`; train 712, test 179).
- Модель: `LogisticRegression(max_iter=1000)` у `Pipeline` з preprocessing ЛР2 (медіана + індикатор, `StandardScaler`, one-hot). Нових алгоритмів не шукаємо.
- Бал: `predict_proba[:, 1]` — оцінка ймовірності загибелі.
- OOF: зовнішній `StratifiedKFold(5, shuffle=True, random_state=42)`; калібрування — `CalibratedClassifierCV(method="sigmoid", cv=3)` усередині кожного зовнішнього fold-а.
- Ресурсне обмеження (припущення сценарію): екіпаж може супроводити **не більше 50%** пасажирів, `q(t) ≤ 0,50`.

In [2]:
df = load_titanic()
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = stratified_split(X, y)
outer_cv = make_cv()
base_model = Pipeline([("preprocessor", make_preprocessor()),
                       ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
calibrated_model = CalibratedClassifierCV(estimator=clone(base_model), method="sigmoid", cv=3)

fold_id = np.empty(len(X_train), dtype=int)
for fold, (_, va) in enumerate(outer_cv.split(X_train, y_train), 1):
    fold_id[va] = fold
raw_oof = cross_val_predict(base_model, X_train, y_train, cv=outer_cv, method="predict_proba")[:, 1]
cal_oof = cross_val_predict(calibrated_model, X_train, y_train, cv=outer_cv, method="predict_proba")[:, 1]
oof = pd.DataFrame({"object_id": X_train.index, "y_true": y_train.values, "score_raw": raw_oof, "score_calibrated": cal_oof,
                    "fold": fold_id})
oof.to_csv(RESULTS / "oof_scores.csv", index=False)
print("Кожен об'єкт має рівно один OOF-прогноз:", oof["object_id"].is_unique and len(oof) == len(X_train))
print("Розміри fold-ів:", oof["fold"].value_counts().sort_index().tolist())
display(oof.head())

Кожен об'єкт має рівно один OOF-прогноз: True
Розміри fold-ів: [143, 143, 142, 142, 142]


,object_id,y_true,score_raw,score_calibrated,fold
0,853,1,0.195467,0.219040,4
1,708,0,0.643952,0.646500,2
2,663,1,0.675006,0.671554,1
3,34,1,0.936919,0.918013,4
4,557,0,0.120463,0.150787,2


## 3. Калібрування (OOF)

Калібрування перевіряється до вибору порога, бо формула теоретичного порога вимагає каліброваних ймовірностей.

In [3]:
calib_rows = []
for name, score in [("Логістична регресія (без калібрування)", raw_oof), ("Після sigmoid-калібрування", cal_oof)]:
    calib_rows.append({"Варіант": name, "ROC-AUC": roc_auc_score(y_train, score), "Brier score": brier_score_loss(y_train, score),
                       "Середній бал": score.mean(), "Фактична частка y=1": y_train.mean()})
calibration = pd.DataFrame(calib_rows)
calibration.to_csv(RESULTS / "calibration.csv", index=False)
display(calibration.style.format({c: "{:.4f}" for c in calibration.columns if c != "Варіант"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, score, color in [("без калібрування", raw_oof, "#3f7f93"), ("sigmoid", cal_oof, "#c46a5a")]:
    frac, mean_pred = calibration_curve(y_train, score, n_bins=10, strategy="quantile")
    axes[0].plot(mean_pred, frac, marker="o", color=color, label=f"{name}, Brier = {brier_score_loss(y_train, score):.4f}")
    axes[1].hist(score, bins=30, alpha=0.5, color=color, label=name)
axes[0].plot([0, 1], [0, 1], "--", color="gray", label="ідеальне калібрування")
axes[0].set(title="Калібрувальна діаграма (OOF, 10 квантильних bins)", xlabel="Середній прогноз у bin", ylabel="Фактична частка AtRisk")
axes[0].legend(fontsize=8)
axes[1].set(title="Розподіл OOF-балів", xlabel="P(AtRisk = 1)", ylabel="Пасажирів")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURES / "calibration.png", dpi=160, bbox_inches="tight")
plt.show()

bins = pd.qcut(raw_oof, 10, duplicates="drop")
bin_table = pd.DataFrame({"bin": bins, "pred": raw_oof, "y": y_train.values}).groupby("bin", observed=True).agg(
    n=("y", "size"), mean_pred=("pred", "mean"), actual=("y", "mean"))
bin_table["actual − pred"] = bin_table["actual"] - bin_table["mean_pred"]
display(bin_table.style.format({"mean_pred": "{:.3f}", "actual": "{:.3f}", "actual − pred": "{:+.3f}"}))

,Варіант,ROC-AUC,Brier score,Середній бал,Фактична частка y=1
0,Логістична регресія (без калібрування),0.8530,0.1428,0.6158,0.6166
1,Після sigmoid-калібрування,0.8527,0.1428,0.6155,0.6166


,n,mean_pred,actual,actual − pred
bin,,,,
"(0.02, 0.114]",72,0.067,0.042,-0.026
"(0.114, 0.26]",71,0.185,0.127,-0.059
"(0.26, 0.421]",71,0.353,0.394,+0.042
"(0.421, 0.583]",71,0.505,0.620,+0.115
"(0.583, 0.706]",71,0.643,0.676,+0.033
"(0.706, 0.838]",71,0.774,0.803,+0.029
"(0.838, 0.882]",72,0.865,0.847,-0.017
"(0.882, 0.912]",70,0.896,0.857,-0.038
"(0.912, 0.934]",72,0.921,0.889,-0.032


In [4]:
SCORE = cal_oof if calibration.loc[1, "Brier score"] < calibration.loc[0, "Brier score"] else raw_oof
SCORE_NAME = "score_calibrated" if SCORE is cal_oof else "score_raw"
print("Для подальших розрахунків використовується:", SCORE_NAME)

Для подальших розрахунків використовується: score_raw


## 4. Контрольна політика t = 0,5

In [5]:
def evaluate_threshold(y_true, score, threshold, c_fp=1.0, c_fn=1.0):
    y_pred = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    n = len(y_true)
    return {"threshold": threshold, "tn": tn, "fp": fp, "fn": fn, "tp": tp,
            "precision": precision_score(y_true, y_pred, zero_division=0), "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0), "alert_rate": (tp + fp) / n, "actions_per_1000": 1000 * (tp + fp) / n,
            "cost_per_object": (c_fp * fp + c_fn * fn) / n}


baseline_policy = evaluate_threshold(y_train.values, SCORE, 0.5)
display(pd.DataFrame([baseline_policy]).style.format({c: "{:.3f}" for c in ["precision", "recall", "f1", "alert_rate", "cost_per_object"]}))

,threshold,tn,fp,fn,tp,precision,recall,f1,alert_rate,actions_per_1000,cost_per_object
0,0.500000,189,84,55,384,0.821,0.875,0.847,0.657,657.303371,0.195


## 5. Матриця вартостей і сценарії

| Факт \\ рішення | Стандартна евакуація (`ŷ=0`) | Пріоритетний супровід (`ŷ=1`) |
|---|---|---|
| **Вижив (`y=0`)** | **TN**: ресурс не витрачено, наслідків немає. Вартість 0 | **FP**: зайвий супровід, член екіпажу не допоміг іншому. Одиниця — «зайвий супровід» (`c_FP`) |
| **Не вижив (`y=1`)** | **FN**: людина з ризиком без допомоги. Одиниця — «пропущений пасажир у групі ризику» (`c_FN`) | **TP**: допомогу отримав той, кому вона потрібна. Додаткова вартість 0 (бажана дія) |

Усі вартості — **сценарні** припущення (не фактичні й не нормативні): реальну «ціну» пропущеного пасажира не можна встановити з даних, і система ШІ не є її джерелом. Тому аналізуємо три сценарії й неперервну чутливість.

| Сценарій | `c_FP` | `c_FN` | Інтерпретація |
|---|---:|---:|---|
| Симетричний (контрольний) | 1 | 1 | помилки рівноцінні |
| Реалістичний | 1 | 5 | пропущений пасажир із ризиком у 5 разів гірший за зайвий супровід |
| Стресовий | 1 | 20 | будь-який пропуск неприйнятний |

In [6]:
SCENARIOS = {"Симетричний": (1.0, 1.0), "Реалістичний": (1.0, 5.0), "Стресовий": (1.0, 20.0)}
MAIN_SCENARIO = "Реалістичний"
CAPACITY = 0.50
THRESHOLDS = np.round(np.arange(0.01, 1.00, 0.01), 2)

grid_rows = []
for t in THRESHOLDS:
    row = evaluate_threshold(y_train.values, SCORE, t)
    for name, (c_fp, c_fn) in SCENARIOS.items():
        row[f"cost_{name}"] = (c_fp * row["fp"] + c_fn * row["fn"]) / len(y_train)
    grid_rows.append(row)
grid = pd.DataFrame(grid_rows)
grid.to_csv(RESULTS / "threshold_grid.csv", index=False)
print("Порогів у сітці:", len(grid), "| діапазон балів:", round(SCORE.min(), 3), "–", round(SCORE.max(), 3))

edge = pd.DataFrame([evaluate_threshold(y_train.values, SCORE, t) for t in [0.0, 1.0, float(SCORE[0])]])
edge.insert(0, "випадок", ["t = 0 (усім супровід)", "t = 1 (нікому)", f"t = бал об'єкта 0 ({SCORE[0]:.4f})"])
display(edge[["випадок", "tp", "fp", "fn", "tn", "alert_rate"]])
print("Об'єкт 0 з t = його бал отримує позитивне рішення:", (SCORE[0] >= SCORE[0]))

Порогів у сітці: 99 | діапазон балів: 0.021 – 0.997


,випадок,tp,fp,fn,tn,alert_rate
0,t = 0 (усім супровід),439,273,0,0,1.000000
1,t = 1 (нікому),0,0,439,273,0.000000
2,t = бал об'єкта 0 (0.1955),435,164,4,109,0.841292


Об'єкт 0 з t = його бал отримує позитивне рішення: True


In [7]:
key_points = grid[grid["threshold"].isin([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8])]
display(key_points[["threshold", "tp", "fp", "fn", "tn", "precision", "recall", "alert_rate", "cost_Симетричний",
                    "cost_Реалістичний", "cost_Стресовий"]].style.format(
    {"threshold": "{:.2f}", "precision": "{:.3f}", "recall": "{:.3f}", "alert_rate": "{:.3f}", "cost_Симетричний": "{:.3f}",
     "cost_Реалістичний": "{:.3f}", "cost_Стресовий": "{:.3f}"}))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].plot(grid["threshold"], grid["precision"], label="Precision", color="#3f7f93")
axes[0].plot(grid["threshold"], grid["recall"], label="Recall", color="#c46a5a")
axes[0].plot(grid["threshold"], grid["f1"], label="F1", color="#6a8f3f")
axes[0].set(title="Precision, Recall, F1 від порога (OOF)", xlabel="Поріг t", ylabel="Значення", ylim=(0, 1.02))
axes[0].legend()
axes[1].plot(grid["threshold"], grid["alert_rate"], color="#8a6aa8", label="q(t) — частка супроводів")
axes[1].axhline(CAPACITY, color="black", ls="--", label=f"місткість {CAPACITY:.0%}")
axes[1].axhline(y_train.mean(), color="gray", ls=":", label=f"частка AtRisk {y_train.mean():.1%}")
axes[1].set(title="Операційне навантаження від порога (OOF)", xlabel="Поріг t", ylabel="q(t)")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURES / "threshold_metrics.png", dpi=160, bbox_inches="tight")
plt.show()

,threshold,tp,fp,fn,tn,precision,recall,alert_rate,cost_Симетричний,cost_Реалістичний,cost_Стресовий
9,0.10,437,213,2,60,0.672,0.995,0.913,0.302,0.313,0.355
19,0.20,434,163,5,110,0.727,0.989,0.838,0.236,0.264,0.369
29,0.30,422,135,17,138,0.758,0.961,0.782,0.213,0.309,0.667
39,0.40,405,106,34,167,0.793,0.923,0.718,0.197,0.388,1.104
49,0.50,384,84,55,189,0.821,0.875,0.657,0.195,0.504,1.663
59,0.60,353,64,86,209,0.847,0.804,0.586,0.211,0.694,2.506
69,0.70,310,49,129,224,0.864,0.706,0.504,0.250,0.975,3.692
79,0.80,266,40,173,233,0.869,0.606,0.430,0.299,1.271,4.916


## 6–7. Порівняння сценаріїв і ресурсне обмеження

Для каліброваних ймовірностей теоретичний поріг $t^* = c_{FP} / (c_{FP} + c_{FN})$. Емпіричний поріг — мінімум $\bar C(t)$ на OOF-сітці; допустимий — мінімум серед $q(t) \le 0,50$.

In [8]:
scenario_rows = []
for name, (c_fp, c_fn) in SCENARIOS.items():
    col = f"cost_{name}"
    t_star = c_fp / (c_fp + c_fn)
    best = grid.loc[grid[col].idxmin()]
    feasible = grid[grid["alert_rate"] <= CAPACITY]
    best_f = feasible.loc[feasible[col].idxmin()]
    scenario_rows.append({"Сценарій": name, "c_FN/c_FP": c_fn / c_fp, "t*": t_star, "t_min C": best["threshold"],
                          "C(t_min)": best[col], "q(t_min)": best["alert_rate"], "Recall(t_min)": best["recall"],
                          "Precision(t_min)": best["precision"], "t_op (q ≤ 50%)": best_f["threshold"], "C(t_op)": best_f[col],
                          "q(t_op)": best_f["alert_rate"], "Recall(t_op)": best_f["recall"], "Precision(t_op)": best_f["precision"]})
scenarios = pd.DataFrame(scenario_rows)
scenarios.to_csv(RESULTS / "scenarios.csv", index=False)
display(scenarios.style.format({c: "{:.3f}" for c in scenarios.columns if c not in ("Сценарій",)}))

fig, ax = plt.subplots(figsize=(10, 5))
for (name, _), color in zip(SCENARIOS.items(), ["#3f7f93", "#c46a5a", "#8a6aa8"]):
    ax.plot(grid["threshold"], grid[f"cost_{name}"], color=color, label=f"{name} (1:{SCENARIOS[name][1]:.0f})")
    r = scenarios.set_index("Сценарій").loc[name]
    ax.scatter(r["t_min C"], r["C(t_min)"], color=color, s=60, zorder=3)
    ax.scatter(r["t_op (q ≤ 50%)"], r["C(t_op)"], color=color, s=80, marker="s", edgecolor="black", zorder=3)
infeasible_until = grid.loc[grid["alert_rate"] <= CAPACITY, "threshold"].min()
ax.axvspan(0, infeasible_until, color="gray", alpha=0.12, label=f"q(t) > {CAPACITY:.0%} (недопустимо)")
ax.set(yscale="log", title="Середня вартість рішень від порога (кола — мінімум, квадрати — мінімум за місткості)",
       xlabel="Поріг t", ylabel="C̄(t), умовних од. / пасажира (log)")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES / "cost_curves.png", dpi=160, bbox_inches="tight")
plt.show()

,Сценарій,c_FN/c_FP,t*,t_min C,C(t_min),q(t_min),Recall(t_min),Precision(t_min),t_op (q ≤ 50%),C(t_op),q(t_op),Recall(t_op),Precision(t_op)
0,Симетричний,1.000,0.500,0.420,0.195,0.699,0.909,0.801,0.710,0.253,0.499,0.699,0.865
1,Реалістичний,5.000,0.167,0.190,0.254,0.846,0.993,0.724,0.710,0.994,0.499,0.699,0.865
2,Стресовий,20.000,0.048,0.190,0.317,0.846,0.993,0.724,0.710,3.775,0.499,0.699,0.865


In [9]:
main = scenarios.set_index("Сценарій").loc[MAIN_SCENARIO]
T_OP = float(main["t_op (q ≤ 50%)"])
c_fp_main, c_fn_main = SCENARIOS[MAIN_SCENARIO]
policies = pd.DataFrame([
    {"Політика": "Контрольна t = 0,5", **evaluate_threshold(y_train.values, SCORE, 0.5, c_fp_main, c_fn_main)},
    {"Політика": "Мінімум вартості", **evaluate_threshold(y_train.values, SCORE, float(main["t_min C"]), c_fp_main, c_fn_main)},
    {"Політика": "Мінімум за місткості 50%", **evaluate_threshold(y_train.values, SCORE, T_OP, c_fp_main, c_fn_main)},
])
policies.to_csv(RESULTS / "candidate_policies.csv", index=False)
display(policies[["Політика", "threshold", "tp", "fp", "fn", "tn", "precision", "recall", "alert_rate", "actions_per_1000",
                  "cost_per_object"]].style.format({"threshold": "{:.2f}", "precision": "{:.3f}", "recall": "{:.3f}",
                                                    "alert_rate": "{:.3f}", "actions_per_1000": "{:.0f}", "cost_per_object": "{:.3f}"}))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
feasible_mask = grid["alert_rate"] <= CAPACITY
axes[0].scatter(grid.loc[~feasible_mask, "recall"], grid.loc[~feasible_mask, "precision"], s=14, color="#bbbbbb", label="q > 50%")
axes[0].scatter(grid.loc[feasible_mask, "recall"], grid.loc[feasible_mask, "precision"], s=14, color="#3f7f93", label="q ≤ 50%")
for _, r in policies.iterrows():
    axes[0].scatter(r["recall"], r["precision"], s=110, marker="*", zorder=3, label=f"{r['Політика']} (t={r['threshold']:.2f})")
axes[0].set(title="Робочі точки на Precision–Recall (OOF)", xlabel="Recall", ylabel="Precision")
axes[0].legend(fontsize=8)
sns.histplot(x=SCORE, hue=y_train.map({0: "вижив", 1: "не вижив"}).values, bins=30, element="step", ax=axes[1],
             palette={"вижив": "#3f7f93", "не вижив": "#c46a5a"})
axes[1].axvline(T_OP, color="black", ls="--", label=f"t = {T_OP:.2f}")
axes[1].set(title="Розподіл OOF-балів за фактичним класом", xlabel="P(AtRisk = 1)", ylabel="Пасажирів")
fig.tight_layout()
fig.savefig(FIGURES / "operating_point.png", dpi=160, bbox_inches="tight")
plt.show()

,Політика,threshold,tp,fp,fn,tn,precision,recall,alert_rate,actions_per_1000,cost_per_object
0,"Контрольна t = 0,5",0.50,384,84,55,189,0.821,0.875,0.657,657,0.504
1,Мінімум вартості,0.19,436,166,3,107,0.724,0.993,0.846,846,0.254
2,Мінімум за місткості 50%,0.71,307,48,132,225,0.865,0.699,0.499,499,0.994


**Жорстка квота.** Якщо 50% — жорстка межа (наприклад, фізична кількість членів екіпажу), фіксований поріг її не гарантує: на новій партії частка балів вище порога може бути іншою. Тоді використовується `top_k_decisions` (`src/decision_policy.py`): у кожній партії позначаються рівно `k` пасажирів із найбільшим балом, ties розв'язуються за зростанням `PassengerId`, а надлишок понад `k` іде в чергу на стандартну евакуацію з повідомленням черговому офіцеру.

## 8. Аналіз чутливості та стійкості

In [10]:
ratios = np.array([1, 1.5, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 25, 30])
sens_rows = []
feasible = grid[grid["alert_rate"] <= CAPACITY]
for r in ratios:
    cost = (grid["fp"] + r * grid["fn"]) / len(y_train)
    cost_f = (feasible["fp"] + r * feasible["fn"]) / len(y_train)
    i, j = cost.idxmin(), cost_f.idxmin()
    sens_rows.append({"c_FN/c_FP": r, "t*": 1 / (1 + r), "t_min": grid.loc[i, "threshold"], "q(t_min)": grid.loc[i, "alert_rate"],
                      "t_op": feasible.loc[j, "threshold"], "q(t_op)": feasible.loc[j, "alert_rate"]})
sensitivity = pd.DataFrame(sens_rows)
sensitivity.to_csv(RESULTS / "sensitivity.csv", index=False)
display(sensitivity.style.format({"c_FN/c_FP": "{:g}", "t*": "{:.3f}", "t_min": "{:.2f}", "q(t_min)": "{:.3f}", "t_op": "{:.2f}", "q(t_op)": "{:.3f}"}))

rng = np.random.default_rng(RANDOM_STATE)
boot_thresholds, boot_unconstrained = [], []
yv = y_train.values
for _ in range(500):
    idx = rng.integers(0, len(yv), len(yv))
    yb, sb = yv[idx], SCORE[idx]
    pred = sb[None, :] >= THRESHOLDS[:, None]
    fp = ((pred == 1) & (yb == 0)).sum(1)
    fn = ((pred == 0) & (yb == 1)).sum(1)
    q = pred.mean(1)
    cost = (c_fp_main * fp + c_fn_main * fn) / len(yb)
    boot_unconstrained.append(THRESHOLDS[cost.argmin()])
    cost_f = np.where(q <= CAPACITY, cost, np.inf)
    boot_thresholds.append(THRESHOLDS[cost_f.argmin()])
boot = pd.DataFrame({"t_op": boot_thresholds, "t_min": boot_unconstrained})
boot.to_csv(RESULTS / "bootstrap_thresholds.csv", index=False)
display(boot.quantile([0.05, 0.5, 0.95]).style.format("{:.3f}"))

near = grid[grid[f"cost_{MAIN_SCENARIO}"] <= grid[f"cost_{MAIN_SCENARIO}"].min() * 1.01]
near_f = feasible[(feasible[f"cost_{MAIN_SCENARIO}"] <= feasible[f"cost_{MAIN_SCENARIO}"].min() * 1.01)]
print(f"Майже рівноцінний діапазон (≤ +1% від мінімуму) без обмеження: {near['threshold'].min():.2f}–{near['threshold'].max():.2f}")
print(f"Те саме серед допустимих (q ≤ 50%): {near_f['threshold'].min():.2f}–{near_f['threshold'].max():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].plot(sensitivity["c_FN/c_FP"], sensitivity["t*"], marker="o", label="теоретичний t*", color="gray")
axes[0].plot(sensitivity["c_FN/c_FP"], sensitivity["t_min"], marker="o", label="емпіричний мінімум", color="#c46a5a")
axes[0].plot(sensitivity["c_FN/c_FP"], sensitivity["t_op"], marker="s", label="мінімум за q ≤ 50%", color="#3f7f93")
axes[0].set(xscale="log", title="Чутливість порога до c_FN/c_FP", xlabel="c_FN / c_FP (log)", ylabel="Поріг")
axes[0].legend()
ax2 = axes[0].twinx()
ax2.plot(sensitivity["c_FN/c_FP"], sensitivity["q(t_min)"], ls=":", color="#c46a5a")
ax2.set_ylabel("q(t_min), пунктир")
ax2.grid(False)
axes[1].hist(boot["t_op"], bins=np.arange(0.3, 0.8, 0.01), color="#3f7f93", alpha=0.7, label="t_op (q ≤ 50%)")
axes[1].hist(boot["t_min"], bins=np.arange(0.0, 0.5, 0.01), color="#c46a5a", alpha=0.7, label="t_min (без обмеження)")
axes[1].set(title="Bootstrap (500) вибраного порога, реалістичний сценарій", xlabel="Поріг", ylabel="Повторів")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURES / "sensitivity.png", dpi=160, bbox_inches="tight")
plt.show()

,c_FN/c_FP,t*,t_min,q(t_min),t_op,q(t_op)
0,1,0.500,0.42,0.699,0.71,0.499
1,1.5,0.400,0.38,0.735,0.71,0.499
2,2,0.333,0.21,0.829,0.71,0.499
3,3,0.250,0.21,0.829,0.71,0.499
4,4,0.200,0.21,0.829,0.71,0.499
5,5,0.167,0.19,0.846,0.71,0.499
6,6,0.143,0.19,0.846,0.71,0.499
7,8,0.111,0.19,0.846,0.71,0.499
8,10,0.091,0.19,0.846,0.71,0.499
9,12,0.077,0.19,0.846,0.71,0.499


,t_op,t_min
0.050000,0.670,0.190
0.500000,0.710,0.210
0.950000,0.750,0.231


Майже рівноцінний діапазон (≤ +1% від мінімуму) без обмеження: 0.19–0.21
Те саме серед допустимих (q ≤ 50%): 0.71–0.71


## 9. Політика з ручним переглядом (три зони)

Верхні 40% пасажирів за балом — автоматичний пріоритетний супровід, наступні 15% — **ручний перегляд** черговим офіцером (наприклад, з урахуванням того, чи пасажир з дітьми, чи має обмеження мобільності — інформації, якої немає в моделі), решта 45% — стандартна евакуація. Пороги визначено лише за квантилями OOF-балів.

In [11]:
T_HIGH = float(np.round(np.quantile(SCORE, 1 - 0.40), 3))
T_LOW = float(np.round(np.quantile(SCORE, 1 - 0.55), 3))
three_zone = DecisionPolicy(threshold_low=T_LOW, threshold_high=T_HIGH)
zones = pd.DataFrame({"score": SCORE, "y": y_train.values, "action": three_zone.decide_batch(SCORE)})
zone_table = zones.groupby("action").agg(n=("y", "size"), share=("y", lambda s: len(s) / len(zones)), mean_score=("score", "mean"),
                                         actual_rate=("y", "mean")).loc[["priority_assistance", "manual_review", "standard"]]
zone_table["Навантаження на 1000 пасажирів"] = zone_table["share"] * 1000
zone_table.to_csv(RESULTS / "three_zone_policy.csv")
print(f"t_low = {T_LOW}, t_high = {T_HIGH}")
display(zone_table.style.format({"share": "{:.1%}", "mean_score": "{:.3f}", "actual_rate": "{:.3f}", "Навантаження на 1000 пасажирів": "{:.0f}"}))

single = evaluate_threshold(y_train.values, SCORE, T_OP, c_fp_main, c_fn_main)
in_priority = zones["action"] == "priority_assistance"
in_review = zones["action"] == "manual_review"
compare_zone = pd.DataFrame([
    {"Політика": f"Один поріг t = {T_OP:.2f}", "Автосупровід": single["alert_rate"], "Ручний перегляд": 0.0,
     "FN поза супроводом (частка всіх AtRisk)": single["fn"] / y_train.sum()},
    {"Політика": f"Три зони ({T_LOW}/{T_HIGH})", "Автосупровід": in_priority.mean(), "Ручний перегляд": in_review.mean(),
     "FN поза супроводом (частка всіх AtRisk)": ((zones["action"] == "standard") & (zones["y"] == 1)).sum() / y_train.sum()},
])
display(compare_zone.style.format({c: "{:.3f}" for c in compare_zone.columns if c != "Політика"}))

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.histplot(x=SCORE, hue=y_train.map({0: "вижив", 1: "не вижив"}).values, bins=40, element="step", ax=ax,
             palette={"вижив": "#3f7f93", "не вижив": "#c46a5a"})
ax.axvspan(0, T_LOW, color="#3f7f93", alpha=0.08)
ax.axvspan(T_LOW, T_HIGH, color="#d49a3a", alpha=0.15)
ax.axvspan(T_HIGH, 1, color="#c46a5a", alpha=0.08)
for x, label in [(T_LOW / 2, "стандартна"), ((T_LOW + T_HIGH) / 2, "ручний\nперегляд"), ((T_HIGH + 1) / 2, "пріоритетний\nсупровід")]:
    ax.text(x, ax.get_ylim()[1] * 0.85, label, ha="center", fontsize=9)
ax.set(title="Три зони операційного рішення (OOF-бали train)", xlabel="P(AtRisk = 1)", ylabel="Пасажирів")
fig.tight_layout()
fig.savefig(FIGURES / "three_zones.png", dpi=160, bbox_inches="tight")
plt.show()

t_low = 0.646, t_high = 0.838


,n,share,mean_score,actual_rate,Навантаження на 1000 пасажирів
action,,,,,
priority_assistance,285,40.0%,0.909,0.877,400
manual_review,107,15.0%,0.740,0.804,150
standard,320,44.9%,0.313,0.322,449


,Політика,Автосупровід,Ручний перегляд,FN поза супроводом (частка всіх AtRisk)
0,Один поріг t = 0.71,0.499,0.000,0.301
1,Три зони (0.646/0.838),0.400,0.150,0.235


Ефективність ручного перегляду невідома, тому три-зонна політика **не** оцінюється вартістю: порівнюється лише навантаження і те, яка частка пасажирів із ризиком залишається поза будь-якою увагою. Для економічного порівняння потрібні дані про точність і тривалість ручного перегляду.

## 10–11. Політика як компонент: конфігурація і тести

Основна політика — **одно-порогова** `t_op` (поріг `threshold_low = threshold_high`, тобто зони ручного перегляду немає); три-зонні пороги записані в конфігурацію як альтернативний варіант. Код вибору дії (`src/decision_policy.py`) читає пороги лише з конфігурації.

In [12]:
config = {
    "policy_id": "titanic-evacuation-priority",
    "version": "1.0.0",
    "score_semantics": "sigmoid-calibrated P(AtRisk = 1 | x) from LogisticRegression (lab02 pipeline)" if SCORE_NAME == "score_calibrated"
    else "P(AtRisk = 1 | x) from LogisticRegression (lab02 pipeline), not recalibrated",
    "decision_rule": "score >= threshold",
    "thresholds": {"low": T_OP, "high": T_OP},
    "actions": ["standard", "manual_review", "priority_assistance"],
    "alternative_three_zone": {"low": T_LOW, "high": T_HIGH, "shares": {"priority": 0.40, "review": 0.15, "standard": 0.45}},
    "hard_quota_fallback": "top_k_decisions(k = floor(0.50 * batch_size)), ties by PassengerId ascending",
    "cost_scenario": {"name": MAIN_SCENARIO, "c_fp": c_fp_main, "c_fn": c_fn_main, "units": "scenario units (1 = one unnecessary escort)",
                      "source": "scenario assumption, not measured"},
    "capacity": {"max_alert_rate": CAPACITY, "type": "soft target"},
    "selected_on": "OOF predictions on D_train (712 passengers), StratifiedKFold(5, shuffle, random_state=42)",
    "data": "data/titanic.csv (Kaggle train.csv), split 80/20 stratified, random_state=42",
    "model_version": "lab02 LogisticRegression(max_iter=1000) + preprocessing from src/titanic_workflow.py",
    "random_state": RANDOM_STATE,
    "created": "2026-10-03",
    "owner": "Maksym Lapko",
    "review_conditions": ["alert rate outside 0.40–0.60 on a new batch", "Brier score on labelled data > 0.16",
                          "change of crew capacity or cost assumptions", "model retraining"],
}
(CONFIGS / "policy_config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
policy = DecisionPolicy.from_config(CONFIGS / "policy_config.json")
print(policy)

tests = subprocess.run([sys.executable, "-m", "unittest", "-v", "tests/test_decision_policy.py"], cwd=WORK_DIR,
                       capture_output=True, text=True)
print(tests.stderr)

DecisionPolicy(threshold_low=0.71, threshold_high=0.71, actions=('standard', 'manual_review', 'priority_assistance'))
test_above_high_threshold_is_priority (tests.test_decision_policy.DecisionPolicyTest.test_above_high_threshold_is_priority) ... ok
test_batch_keeps_order (tests.test_decision_policy.DecisionPolicyTest.test_batch_keeps_order) ... ok
test_below_low_threshold_is_standard (tests.test_decision_policy.DecisionPolicyTest.test_below_low_threshold_is_standard) ... ok
test_between_thresholds_is_manual_review (tests.test_decision_policy.DecisionPolicyTest.test_between_thresholds_is_manual_review) ... ok
test_config_file_is_consistent_with_code (tests.test_decision_policy.DecisionPolicyTest.test_config_file_is_consistent_with_code) ... ok
test_config_round_trip (tests.test_decision_policy.DecisionPolicyTest.test_config_round_trip) ... ok
test_equal_to_high_threshold_is_priority (tests.test_decision_policy.DecisionPolicyTest.test_equal_to_high_threshold_is_priority) ... ok
test_equa

## 12–13. Фіксація рішення і фінальне оцінювання (test відкривається один раз)

Запис рішення до відкриття test: модель — LR + preprocessing ЛР2 з тим самим варіантом калібрування, що обраний у п.3; правило `score >= t_op`; сценарій `c_FP=1, c_FN=5`; м'яка місткість 50%; метрики — confusion matrix, Precision, Recall, частка дій, Brier score, середня вартість. Очікуване навантаження — близько `q(t_op)` на OOF.

In [13]:
final_model = (clone(calibrated_model) if SCORE_NAME == "score_calibrated" else clone(base_model)).fit(X_train, y_train)
test_score = final_model.predict_proba(X_test)[:, 1]
actions = policy.decide_batch(test_score)
test_pred = np.array([a == "priority_assistance" for a in actions], dtype=int)
tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()
final = pd.DataFrame([
    {"Набір": "OOF train", **evaluate_threshold(y_train.values, SCORE, T_OP, c_fp_main, c_fn_main),
     "brier": brier_score_loss(y_train, SCORE), "roc_auc": roc_auc_score(y_train, SCORE)},
    {"Набір": "Test", **evaluate_threshold(y_test.values, test_score, T_OP, c_fp_main, c_fn_main),
     "brier": brier_score_loss(y_test, test_score), "roc_auc": roc_auc_score(y_test, test_score)},
])
final.to_csv(RESULTS / "final_test.csv", index=False)
pd.DataFrame({"PassengerId": X_test.index, "y_true": y_test.values, "score": test_score, "action": actions}).to_csv(
    RESULTS / "test_decisions.csv", index=False)
display(final[["Набір", "threshold", "tn", "fp", "fn", "tp", "precision", "recall", "f1", "alert_rate", "brier", "roc_auc",
               "cost_per_object"]].style.format({c: "{:.3f}" for c in ["precision", "recall", "f1", "alert_rate", "brier", "roc_auc",
                                                                      "cost_per_object"]} | {"threshold": "{:.2f}"}))

,Набір,threshold,tn,fp,fn,tp,precision,recall,f1,alert_rate,brier,roc_auc,cost_per_object
0,OOF train,0.71,225,48,132,307,0.865,0.699,0.773,0.499,0.143,0.853,0.994
1,Test,0.71,56,13,27,83,0.865,0.755,0.806,0.536,0.146,0.825,0.827
